# ESM Embedding Exploration

*Write intro here explaining purpose

In [1]:
# imports
from transformers import AutoTokenizer, TFEsmModel 
import tensorflow as tf
from Bio import SeqIO
import sklearn
import plotly.express as px
import os
import numpy as np

# embedding directory
output_directory = "../data/variant_embeddings/"

# load model
hug_face_id = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(hug_face_id)
model = TFEsmModel.from_pretrained(hug_face_id)

# sanity check
inputs = tokenizer("Hello, my dog is cute", return_tensors="tf")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state

print(last_hidden_states, outputs)

2024-11-01 11:02:06.871018: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-01 11:02:06.881665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1730476926.893598   88605 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1730476926.897334   88605 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-01 11:02:06.909493: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

tf.Tensor(
[[[ 0.05059963  0.10085721 -0.10023749 ... -0.12190716  0.40376827
    0.08775394]
  [ 0.26290226  0.02058962 -0.16462716 ... -0.05984128  0.3401519
    0.15950891]
  [ 0.13614741  0.17050998  0.10192016 ... -0.17278999  0.32610363
    0.48546052]
  ...
  [ 0.28022113  0.06663495 -0.09451015 ... -0.11177716  0.1806937
    0.5831055 ]
  [ 0.19588457  0.14026381  0.07719216 ... -0.14424258  0.22779934
    0.39744595]
  [ 0.5040827   0.16508391 -0.00226392 ... -0.24705702  0.12229864
    0.42984688]]], shape=(1, 8, 1280), dtype=float32) TFBaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=<tf.Tensor: shape=(1, 8, 1280), dtype=float32, numpy=
array([[[ 0.05059963,  0.10085721, -0.10023749, ..., -0.12190716,
          0.40376827,  0.08775394],
        [ 0.26290226,  0.02058962, -0.16462716, ..., -0.05984128,
          0.3401519 ,  0.15950891],
        [ 0.13614741,  0.17050998,  0.10192016, ..., -0.17278999,
          0.32610363,  0.48546052],
        ...,
        [ 0

In [2]:
# load heme variant sequences
data_path = "../data/Human_Hb_variants.fasta"
variant_entry = []
with open(data_path, "r") as file:
    for seq in SeqIO.parse(file, "fasta"):
        variant_entry.append(seq)

print("Sample Output:")
print(variant_entry[0].id)
print(variant_entry[0].seq)
print("Number of entries:")
print(len(variant_entry))


Sample Output:
alpha1
MGLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
Number of entries:
1449


In [3]:
# check duplicates / tag their indices
seqs = []
duplicates = []
for i, entry in enumerate(variant_entry):
    sequence = str(entry.seq)
    duplicates = []
    if sequence not in seqs:
        seqs.append(sequence)
    else:
        duplicates.append(i)
        print(f"Duplicate sequence found at index {i}")

print(f"Number of duplicates: {len(duplicates)}")
seqs = list(set(seqs))
print(f"Number of unique sequences: {len(seqs)}")

Duplicate sequence found at index 1
Duplicate sequence found at index 3
Duplicate sequence found at index 5
Duplicate sequence found at index 7
Duplicate sequence found at index 9
Duplicate sequence found at index 11
Duplicate sequence found at index 13
Duplicate sequence found at index 15
Duplicate sequence found at index 17
Duplicate sequence found at index 19
Duplicate sequence found at index 21
Duplicate sequence found at index 23
Duplicate sequence found at index 25
Duplicate sequence found at index 27
Duplicate sequence found at index 29
Duplicate sequence found at index 31
Duplicate sequence found at index 34
Duplicate sequence found at index 37
Duplicate sequence found at index 39
Duplicate sequence found at index 41
Duplicate sequence found at index 43
Duplicate sequence found at index 45
Duplicate sequence found at index 47
Duplicate sequence found at index 49
Duplicate sequence found at index 52
Duplicate sequence found at index 54
Duplicate sequence found at index 56
Duplic

In [4]:
print(set([variant_entry[i].id for i,entry in enumerate(variant_entry)]))

{'750', '751', 'delta', 'Ggamma', '742', '3057', '743', '745', '747', '2807', 'beta', 'Agamma', 'alpha1', '746', 'alpha2', '744'}


Marengo-Rowe AJ. Structure-function relations of human hemoglobins. Proc (Bayl Univ Med Cent). 2006 Jul;19(3):239-45. doi: 10.1080/08998280.2006.11928171. PMID: 17252042; PMCID: PMC1484532.

"The alpha chain of all human hemoglobins, embryonic and adult, is the same. The non-alpha chains include the beta chain of normal adult hemoglobin (α2β2), the gamma chain of fetal hemoglobin (α2β2), and the delta chain of HbA2. In some variants, the gamma genes are duplicated, giving rise to two kinds of gamma chains."

In [5]:
binned_seqs = {'2807':[], 
               '751':[], 
               '745':[],
               '750':[], 
               '747':[],
               '3057':[], 
               'Agamma':[], 
               '746':[], 
               '743':[], 
               'beta':[], 
               'alpha1':[], 
               '742':[], 
               '744':[], 
               'delta':[], 
               'Ggamma':[], 
               'alpha2':[]}

for i in variant_entry:
    binned_seqs[str(i.id)].append(str(i.seq))

for x in binned_seqs:
    print(f"Number of sequences in {x}: {len(binned_seqs[x])}")

Number of sequences in 2807: 1
Number of sequences in 751: 1
Number of sequences in 745: 2
Number of sequences in 750: 1
Number of sequences in 747: 1
Number of sequences in 3057: 2
Number of sequences in Agamma: 42
Number of sequences in 746: 1
Number of sequences in 743: 1
Number of sequences in beta: 563
Number of sequences in alpha1: 295
Number of sequences in 742: 1
Number of sequences in 744: 1
Number of sequences in delta: 91
Number of sequences in Ggamma: 72
Number of sequences in alpha2: 374


Treating Numerical Entries as aberrent, cluster variant embeddings by chain...

In [6]:
var_embed_dict = {
    "Agamma": list(set(binned_seqs['Agamma'])),
    "Ggamma": list(set(binned_seqs['Ggamma'])),
    "alpha1": list(set(binned_seqs['alpha1'])),
    "alpha2": list(set(binned_seqs['alpha2'])),
    "beta": list(set(binned_seqs['beta'])),
    "delta": list(set(binned_seqs['delta'])),
}
del binned_seqs

for i in var_embed_dict:
    print(f"Number of sequences in {i}: {len(var_embed_dict[i])}")

Number of sequences in Agamma: 42
Number of sequences in Ggamma: 72
Number of sequences in alpha1: 295
Number of sequences in alpha2: 374
Number of sequences in beta: 563
Number of sequences in delta: 91


In [7]:
# generate embeddings
os.makedirs(output_directory, exist_ok=True)

max_len = max([len(s) for s in seqs])
print(f"Max sequence length: {max_len}")

for i in var_embed_dict:
    print(f"Processing {i}")
#    with open(output_directory + f"{i}.npy", "wb") as file:
#        writer = csv.writer(file)
    count = 0
    for j in var_embed_dict[i]:
            input = tokenizer(j, return_tensors="tf", padding=True, truncation=True, max_length=max_len)
            output = model(**input)
            os.makedirs(output_directory + f"{i}/", exist_ok=True)
            with open(output_directory + f"{i}/{count}.npy", "wb") as file:
                np.save(file, np.array(output.last_hidden_state))
            count += 1
    print(f"Number of sequences processed: {count}")
    print(f"----- Embeddings for {i} saved -----")

Max sequence length: 148
Processing Agamma
Number of sequences processed: 42
----- Embeddings for Agamma saved -----
Processing Ggamma
Number of sequences processed: 72
----- Embeddings for Ggamma saved -----
Processing alpha1
Number of sequences processed: 295
----- Embeddings for alpha1 saved -----
Processing alpha2
Number of sequences processed: 374
----- Embeddings for alpha2 saved -----
Processing beta
Number of sequences processed: 563
----- Embeddings for beta saved -----
Processing delta
Number of sequences processed: 91
----- Embeddings for delta saved -----


# Plot UMAPs of Embeddings

In [8]:
# load embeddings
embeddings = {}

for directory in os.listdir(output_directory):
    embeddings[directory] = []
    for file in os.listdir(output_directory + directory):
        with open(output_directory + directory + "/" + file, "rb") as f:
            embeddings[directory].append(np.load(f))

# check shapes
for i in embeddings:
    print(f"Number of embeddings in {i}: {len(embeddings[i])}")
    print(f"Shape of embeddings in {i}: {embeddings[i][0].shape}")

Number of embeddings in delta: 91
Shape of embeddings in delta: (1, 148, 1280)
Number of embeddings in beta: 563
Shape of embeddings in beta: (1, 148, 1280)
Number of embeddings in Agamma: 42
Shape of embeddings in Agamma: (1, 148, 1280)
Number of embeddings in alpha2: 374
Shape of embeddings in alpha2: (1, 144, 1280)
Number of embeddings in alpha1: 295
Shape of embeddings in alpha1: (1, 144, 1280)
Number of embeddings in Ggamma: 72
Shape of embeddings in Ggamma: (1, 148, 1280)


In [9]:
for i in embeddings:
    print(f"----- {i} -----")
    shapes = set([z.shape for z in embeddings[i]])
    print(shapes)

----- delta -----
{(1, 148, 1280)}
----- beta -----
{(1, 148, 1280)}
----- Agamma -----
{(1, 148, 1280)}
----- alpha2 -----
{(1, 144, 1280), (1, 145, 1280)}
----- alpha1 -----
{(1, 144, 1280)}
----- Ggamma -----
{(1, 148, 1280)}


In [10]:
def fix_jagged_2d(embed_list):
    sir_gallahad_the_chaste = []
    # find max length in 2nd dim (sequence length)
    max_len = max([len(arr[0]) for arr in embed_list])
    # append zero vector to embeddings
    for i, arr in enumerate(embed_list):
        if len(arr[0]) < max_len:
            padded = np.append(arr[0], np.zeros((max_len - len(arr[0]), 1280)), axis=0)
            sir_gallahad_the_chaste.append(np.expand_dims(padded, axis=0))
        else:
            sir_gallahad_the_chaste.append(arr)
    return sir_gallahad_the_chaste

for emb in embeddings:
    print("Condensing", emb)
    embeddings[emb] = fix_jagged_2d(embeddings[emb])
    reshape = np.concatenate(embeddings[emb])
    print(f"Reshaped {emb} to {reshape.shape}")
    embeddings[emb] = reshape

Condensing delta
Reshaped delta to (91, 148, 1280)
Condensing beta
Reshaped beta to (563, 148, 1280)
Condensing Agamma
Reshaped Agamma to (42, 148, 1280)
Condensing alpha2
Reshaped alpha2 to (374, 145, 1280)
Condensing alpha1
Reshaped alpha1 to (295, 144, 1280)
Condensing Ggamma
Reshaped Ggamma to (72, 148, 1280)


### ESM-2 Vectors
- To plot on UMAP we must reduce the embeddings to shape: (n-variants, feature_vector)
- The simplest way to achieve this is summing across either the residues, or the 1280-d embeddings for those residues
- Resulting in either a dense sequence representation, or a dense feature representation

In [19]:
# demonstrate embeddings
from umap import UMAP

example = embeddings['beta'][42]
print(example.shape)

umap = UMAP(n_components=3)
sequence_umap = umap.fit_transform(example)
esm_umap = umap.fit_transform(example.T)

sequence_rep = px.scatter_3d(sequence_umap, x=0,y=1,z=2, color=2, title="Sequence Rep")
sequence_rep.update_traces(marker_size=3)
esm_rep = px.scatter_3d(esm_umap, x=0,y=1,z=2, color=2, title="ESM2 Rep")
esm_rep.update_traces(marker_size=3)

sequence_rep.show()
esm_rep.show()

(148, 1280)


### Sequence Representation

In [20]:
for emb in embeddings:
    print("----- Processing -----")
    seq_sum = np.add.reduce(embeddings[emb], axis=-1)
    print(f"Reduced {emb} to {seq_sum.shape}")
    print("Manifold Approximation...")
    umap_2d = UMAP(n_components=2)
    seq_umap = umap_2d.fit_transform(seq_sum)
    print("Plotting...")
    fig = px.scatter(seq_umap, x=0, y=1, title=f"UMAP of {emb} Variants")
    fig.show()

----- Processing -----
Reduced delta to (91, 148)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced beta to (563, 148)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced Agamma to (42, 148)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced alpha2 to (374, 145)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced alpha1 to (295, 144)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced Ggamma to (72, 148)
Manifold Approximation...
Plotting...


### ESM-2 Feature Representation

In [21]:
for emb in embeddings:
    print("----- Processing -----")
    seq_sum = np.add.reduce(embeddings[emb], axis=-2)
    print(f"Reduced {emb} to {seq_sum.shape}")
    print("Manifold Approximation...")
    umap_2d = UMAP(n_components=2)
    seq_umap = umap_2d.fit_transform(seq_sum)
    print("Plotting...")
    fig = px.scatter(seq_umap, x=0, y=1, title=f"UMAP of {emb} Variants")
    fig.show()

----- Processing -----
Reduced delta to (91, 1280)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced beta to (563, 1280)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced Agamma to (42, 1280)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced alpha2 to (374, 1280)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced alpha1 to (295, 1280)
Manifold Approximation...
Plotting...


----- Processing -----
Reduced Ggamma to (72, 1280)
Manifold Approximation...
Plotting...
